# Anggota 6 — Evaluasi Jawaban

Priyambodo Arundaya Satmika · 245150201111011

# 1. Aturan Penilaian

In [1]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path
from statistics import mean

import pandas as pd
from IPython.display import display

AMBANG_F1 = 0.70


POLA_PENANDA_BUTIR = re.compile(r"^\s*(?:\d+(?:\.\d+)*|[a-zA-Z])[.)]\s+")


POLA_TANDA_TEPI = re.compile(r"^[\s.,;:]+|[\s.,;:]+$")


POLA_TOKEN_JAWABAN = re.compile(r"\d+(?:[,.]\d+)?|[a-z]+|%|\+|≥|≤|>|<")


def normalisasi_jawaban(teks: str) -> str:

    if not isinstance(teks, str):
        return ""
    hasil = unicodedata.normalize("NFKC", teks).casefold()
    hasil = POLA_PENANDA_BUTIR.sub("", hasil)
    hasil = hasil.replace("\u2018", "'").replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    hasil = re.sub(r"\s+", " ", hasil)
    return POLA_TANDA_TEPI.sub("", hasil)


def token_jawaban(teks: str) -> list[str]:

    return POLA_TOKEN_JAWABAN.findall(normalisasi_jawaban(teks))


def _berisi_berurutan(jarum: list[str], jerami: list[str]) -> bool:

    if not jarum or len(jarum) > len(jerami):
        return False
    for awal in range(len(jerami) - len(jarum) + 1):
        if jerami[awal : awal + len(jarum)] == jarum:
            return True
    return False


def cocok_persis(jawaban_sistem: str, jawaban_rujukan: str) -> bool:

    return bool(normalisasi_jawaban(jawaban_sistem)) and (
        normalisasi_jawaban(jawaban_sistem) == normalisasi_jawaban(jawaban_rujukan)
    )


def cocok_inti(jawaban_sistem: str, jawaban_rujukan: str) -> bool:

    token_sistem = token_jawaban(jawaban_sistem)
    token_rujukan = token_jawaban(jawaban_rujukan)
    return _berisi_berurutan(token_sistem, token_rujukan) or _berisi_berurutan(
        token_rujukan, token_sistem
    )


def f1_token(jawaban_sistem: str, jawaban_rujukan: str) -> float:

    token_sistem = token_jawaban(jawaban_sistem)
    token_rujukan = token_jawaban(jawaban_rujukan)
    if not token_sistem or not token_rujukan:
        return 0.0
    irisan = Counter(token_sistem) & Counter(token_rujukan)
    jumlah_irisan = sum(irisan.values())
    if jumlah_irisan == 0:
        return 0.0
    presisi = jumlah_irisan / len(token_sistem)
    recall = jumlah_irisan / len(token_rujukan)
    return 2 * presisi * recall / (presisi + recall)


def putusan_jawaban(jawaban_sistem: str, jawaban_rujukan: str, ambang_f1: float = AMBANG_F1) -> dict:

    skor_f1 = round(f1_token(jawaban_sistem, jawaban_rujukan), 4)
    if cocok_persis(jawaban_sistem, jawaban_rujukan):
        tingkat = "COCOK_PERSIS"
    elif cocok_inti(jawaban_sistem, jawaban_rujukan):
        tingkat = "COCOK_INTI"
    elif skor_f1 >= ambang_f1:
        tingkat = "MIRIP_PARAFRASA"
    else:
        tingkat = "TIDAK_COCOK"
    return {"tingkat": tingkat, "skor_f1": skor_f1}


def _ringkasan_per_kunci(hasil: list[dict], kunci: str) -> dict:

    rekap: dict[str, dict] = {}
    for baris in hasil:
        label = baris[kunci] or "TIDAK_DIKENALI"
        sel = rekap.setdefault(
            label,
            {"jumlah": 0, "cocok_persis": 0, "cocok_inti": 0, "mirip_parafrasa": 0, "benar_longgar": 0},
        )
        sel["jumlah"] += 1
        if baris["tingkat"] == "COCOK_PERSIS":
            sel["cocok_persis"] += 1
        if baris["tingkat"] in ("COCOK_PERSIS", "COCOK_INTI"):
            sel["cocok_inti"] += 1
        if baris["tingkat"] == "MIRIP_PARAFRASA":
            sel["mirip_parafrasa"] += 1
        if baris["tingkat"] != "TIDAK_COCOK":
            sel["benar_longgar"] += 1

    for label, sel in rekap.items():
        sel["akurasi_persis"] = round(sel["cocok_persis"] / sel["jumlah"], 4)
        sel["akurasi_inti"] = round(sel["cocok_inti"] / sel["jumlah"], 4)
        sel["akurasi_longgar"] = round(sel["benar_longgar"] / sel["jumlah"], 4)
    return dict(sorted(rekap.items()))


def ringkas(hasil: list[dict], ambang_f1: float = AMBANG_F1) -> dict:

    jumlah = len(hasil)
    hitung = Counter(baris["tingkat"] for baris in hasil)
    cocok_inti = hitung["COCOK_PERSIS"] + hitung["COCOK_INTI"]
    benar_longgar = jumlah - hitung["TIDAK_COCOK"]

    di_top5 = [baris for baris in hasil if baris["bukti_di_top5"]]
    luar_top5 = [baris for baris in hasil if not baris["bukti_di_top5"]]
    benar_inti_top5 = sum(
        1 for baris in di_top5 if baris["tingkat"] in ("COCOK_PERSIS", "COCOK_INTI")
    )
    benar_inti_luar = sum(
        1 for baris in luar_top5 if baris["tingkat"] in ("COCOK_PERSIS", "COCOK_INTI")
    )

    return {
        "jumlah_pertanyaan": jumlah,
        "akurasi_persis": round(hitung["COCOK_PERSIS"] / jumlah, 4) if jumlah else 0.0,
        "akurasi_inti": round(cocok_inti / jumlah, 4) if jumlah else 0.0,
        "akurasi_longgar": round(benar_longgar / jumlah, 4) if jumlah else 0.0,
        "jumlah_persis": hitung["COCOK_PERSIS"],
        "jumlah_cocok_inti": cocok_inti,
        "jumlah_mirip_parafrasa": hitung["MIRIP_PARAFRASA"],
        "jumlah_benar_longgar": benar_longgar,
        "jumlah_tidak_cocok": hitung["TIDAK_COCOK"],
        "f1_token_macro": round(mean(baris["skor_f1"] for baris in hasil), 4) if jumlah else 0.0,
        "ambang_f1_parafrasa": ambang_f1,
        "buktidi_top5": len(di_top5),
        "benar_inti_di_top5": benar_inti_top5,
        "akurasi_inti_di_top5": round(benar_inti_top5 / len(di_top5), 4) if di_top5 else 0.0,
        "bukti_luar_top5": len(luar_top5),
        "benar_inti_luar_top5": benar_inti_luar,
        "metode": "EkstraktorJawaban Hybrid (simbolik, linguistik UDPipe, fallback kalimat) Anggota 4",
        "catatan": "Akurasi ketat memakai kesamaan setelah normalisasi; akurasi inti menambah kecocokan sebagian; akurasi longgar menambah parafrasa dengan F1 token >= ambang.",
    }

Tabel membandingkan jawaban aktual dengan rujukan. Akurasi inti mencakup kecocokan persis dan kecocokan token berurutan. Akurasi longgar juga menerima parafrasa yang memenuhi ambang F1.

# 2. Akurasi dan Hasil Per Soal

In [2]:
with open("../Anggota4/output/hasil_ekstraksi.json", encoding="utf-8") as file:
    hasil_ekstraksi = json.load(file)
with open("../Anggota5_Evaluasi_Retrieval/output/hasil_retrieval.json", encoding="utf-8") as file:
    hasil_retrieval = json.load(file)

ekstraksi_by_id = {baris["id_pertanyaan"]: baris for baris in hasil_ekstraksi}
hasil_penilaian = []
for baris in hasil_retrieval["hasil_per_pertanyaan"]:
    keluaran = ekstraksi_by_id[baris["id_pertanyaan"]]
    peringkat = baris["peringkat_bukti_pertama"]
    hasil_penilaian.append({
        "id_pertanyaan": baris["id_pertanyaan"],
        "penanya": baris["penanya"],
        "pertanyaan": baris["pertanyaan"],
        "tipe_jawaban": keluaran["tipe_jawaban"],
        "jawaban_rujukan": baris["jawaban_rujukan"],
        "jawaban_sistem": keluaran["jawaban"],
        "metode": keluaran["metode"],
        "status": keluaran["status"],
        "passage_id": keluaran["passage_id"],
        "sumber": keluaran["sumber"],
        "peringkat_bukti_pertama": peringkat,
        "bukti_di_top5": peringkat is not None and peringkat <= 5,
        **putusan_jawaban(keluaran["jawaban"], baris["jawaban_rujukan"]),
    })

ringkasan_jawaban = {
    **ringkas(hasil_penilaian),
    "per_tipe_jawaban": _ringkasan_per_kunci(hasil_penilaian, "tipe_jawaban"),
    "per_metode": _ringkasan_per_kunci(hasil_penilaian, "metode"),
}
hasil_evaluasi = {"ringkasan": ringkasan_jawaban, "hasil_per_pertanyaan": hasil_penilaian}

display(pd.DataFrame([ringkasan_jawaban])[
    ["jumlah_pertanyaan", "jumlah_persis", "jumlah_cocok_inti", "jumlah_benar_longgar",
     "akurasi_persis", "akurasi_inti", "akurasi_longgar", "f1_token_macro"]
].round(3))
pd.DataFrame(hasil_penilaian)[
    ["id_pertanyaan", "peringkat_bukti_pertama", "jawaban_sistem", "jawaban_rujukan", "tingkat"]
]

,jumlah_pertanyaan,jumlah_persis,jumlah_cocok_inti,jumlah_benar_longgar,akurasi_persis,akurasi_inti,akurasi_longgar,f1_token_macro
0,20,8,13,15,0.4,0.65,0.75,0.696


,id_pertanyaan,peringkat_bukti_pertama,jawaban_sistem,jawaban_rujukan,tingkat
0,A2-001,3,144 sks,144 sks,COCOK_PERSIS
1,A2-002,1,75 menit,75 menit,COCOK_PERSIS
2,A2-003,19,Format skripsi disusun menurut dokumen Panduan...,Karya ilmiah mahasiswa pendidikan program sarj...,TIDAK_COCOK
3,A2-004,6,"Menurut pedoman penilaian PTIIK, nilai angka (...",B+,TIDAK_COCOK
4,A3-001,2,14 semester,14 semester sejak pertama kali terdaftar sebag...,COCOK_INTI
5,A3-002,1,Paling lambat 1 bulan setelah berakhirnya masa...,Paling lambat 1 bulan setelah masa daftar ulan...,MIRIP_PARAFRASA
6,A3-003,25,Keberhasilan studi mahasiswa dinyatakan dengan...,IP = Σ (K_i × Na_i) / Σ K_i.,TIDAK_COCOK
7,A4-001,1,80%,80%,COCOK_PERSIS
8,A4-002,1,15 menit,15 menit,COCOK_PERSIS
9,A4-003,1,"3,5","3,5",COCOK_PERSIS


Ringkasan menunjukkan hasil untuk 20 soal. Kolom peringkat bukti dapat dipakai untuk melihat apakah kesalahan terjadi sebelum atau sesudah ekstraksi.

# 3. Simpan Penilaian

In [3]:
Path("output").mkdir(exist_ok=True)
with open("output/hasil_evaluasi.json", "w", encoding="utf-8") as file:
    json.dump(hasil_evaluasi, file, ensure_ascii=False, indent=2)

Ekspor memuat putusan dan sumber per soal, serta rekap akurasi menurut tipe jawaban dan metode ekstraksi.